In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.ticker as ticker
from datetime import datetime
import os
import h5py
from modules.func import LINEAR, TANH, RELU, LOGSIG
from modules.model import dual_LA_model
from modules.batch import expand_one_hot_vector,reduct_pred

#MNISTのDL
from keras.datasets import mnist
(train_image, train_label), (test_image, test_label) = mnist.load_data()
train_image = train_image / 255.0
test_image  = test_image / 255.0


In [ ]:
# トライアル数
num_trial  = 2
# 学習条件
num_step   = 1*10**4    # step数
num_test   = 500        # 正答率テスト回数(1回のテストあたりの回数)
num_loop   = 50         # loop回数
# モデルの設定
x1_dim      = [784, 100, 10]  #細胞数
x2_dim      = [784, 100, 10]  #細胞数
num_batch  = 20             # batch数
test_trial = 40     #正答率テスト回数(1回の実行あたりのテスト回数)

lr_x       = 10**(-0.9) # xの学習率(更新率)
lr_weight  = 10**(-1.6) # weightの学習率

# 活性化関数
f = LINEAR #LINEAR, TANH, RELU, LOGSIG

# list
energy_list1        = []
accuracy_list1      = []
cossim_list1        = []
init_cossim_list1   = []
cossim_initB1_list  = []

energy_list2        = []
accuracy_list2      = []
cossim_list2        = []
init_cossim_list2   = []
cossim_initB2_list  = []


for _tri in range(num_trial):   # num_trial回繰り返す

    # モデルの選択
    model = dual_LA_model(x1_dim, x2_dim, num_batch, lr_x, lr_weight, f)

    # 一時listの初期化
    energy_list1_tmp        = []
    accuracy_list1_tmp      = []
    cossim_list1_tmp        = []
    init_cossim_list1_tmp   = []
    cossim_initB1_list_tmp  = []

    energy_list2_tmp        = []
    accuracy_list2_tmp      = []
    cossim_list2_tmp        = []
    init_cossim_list2_tmp   = []
    cossim_initB2_list_tmp  = []

    # 初期ベクトル（flatten）
    B_init1 = (model.U[2]).flatten()
    B_init2 = (model.U[4]).flatten()

    for step in tqdm(range(int((num_step)/num_batch))):                 # num_step回繰り返す

        # PC1更新
        idx    = np.random.randint(0, train_image.shape[0], num_batch)  # randomなMNIST画像の選択
        inputs = train_image[idx].reshape(num_batch , -1)               # inputsにrandomなMNIST画像の代入
        target = expand_one_hot_vector(train_label[idx],num_batch,x1_dim[-1])    # batch処理用にvector変換

        model.initialize_states()                   # 活動の初期化
        model.update_x1(inputs, target, num_loop)    # xの更新
        model.update_weight1()                       # wの更新

        cossim_list1_tmp.append(model.save_cossim1())         #コサイン類似度記録
        energy_list1_tmp.append(model.calculate_energy1())    #エネルギー記録

        # PC2更新
        idx    = np.random.randint(0, train_image.shape[0], num_batch)  # randomなMNIST画像の選択
        inputs = train_image[idx].reshape(num_batch , -1)               # inputsにrandomなMNIST画像の代入
        target = expand_one_hot_vector(train_label[idx],num_batch,x2_dim[-1])    # batch処理用にvector変換

        model.initialize_states()                   # 活動の初期化
        model.update_x2(inputs, target, num_loop)    # xの更新
        model.update_weight2()                       # wの更新

        cossim_list2_tmp.append(model.save_cossim2())         #コサイン類似度記録
        energy_list2_tmp.append(model.calculate_energy2())    #エネルギー記録

        # 初期重みと現在重みのコサイン類似度
        B1_current = (model.U[2]).flatten()
        cossim = np.dot(B_init1, B1_current) / (np.linalg.norm(B_init1) * np.linalg.norm(B1_current))
        cossim_initB1_list_tmp.append(cossim)

        B2_current = (model.U[4]).flatten()
        cossim = np.dot(B_init2, B2_current) / (np.linalg.norm(B_init2) * np.linalg.norm(B2_current))
        cossim_initB2_list_tmp.append(cossim)

        # 正答率評価 PC1
        if (step + 1) % int((num_step)/num_batch/test_trial) == 0 or step == 0:
            cnt = 0
            for _test in range(int(num_test/num_batch)):
                idx    = np.random.randint(0, test_image.shape[0], num_batch)
                inputs = test_image[idx].reshape(num_batch , -1)
                model.initialize_states()
                pred_vec= model.predict1(inputs)
                pred_list = reduct_pred(pred_vec, num_batch, x1_dim[-1])
                cnt += np.count_nonzero(pred_list == test_label[idx])
            accuracy_list1_tmp.append(cnt / num_test)

        # 正答率評価 PC2
        if (step + 1) % int((num_step)/num_batch/test_trial) == 0 or step == 0:
            cnt = 0
            for _test in range(int(num_test/num_batch)):
                idx    = np.random.randint(0, test_image.shape[0], num_batch)
                inputs = test_image[idx].reshape(num_batch , -1)
                model.initialize_states()
                pred_vec= model.predict2(inputs)
                pred_list = reduct_pred(pred_vec, num_batch, x2_dim[-1])
                cnt += np.count_nonzero(pred_list == test_label[idx])
            accuracy_list2_tmp.append(cnt / num_test)

    # 結果の格納
    energy_list1.append(energy_list1_tmp)
    accuracy_list1.append(accuracy_list1_tmp)
    cossim_list1.append(cossim_list1_tmp)
    cossim_initB1_list.append(cossim_initB1_list_tmp)


    energy_list2.append(energy_list2_tmp)
    accuracy_list2.append(accuracy_list2_tmp)
    cossim_list2.append(cossim_list2_tmp)
    cossim_initB2_list.append(cossim_initB2_list_tmp)
    

## 保存

In [ ]:
save_PC = True  # PCの結果を保存するかどうか

# 日時の取得
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
func = f.get_name()
filename = f"{num_trial}trial_{num_step}step_{func}_{timestamp}"

# 保存ディレクトリ
save_dir = f"./Dual_PC_result"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, f"result_{filename}.h5")

with h5py.File(save_path, "w") as h5f:

    grp = h5f.create_group("PC1")
    grp.create_dataset("energy", data=np.array(energy_list1))
    grp.create_dataset("accuracy", data=np.array(accuracy_list1))
    grp.create_dataset("cossim", data=np.array(cossim_list1))
    grp.create_dataset("cossim_initB1", data=np.array(cossim_initB1_list))

    grp = h5f.create_group("PC2")
    grp.create_dataset("energy", data=np.array(energy_list2))
    grp.create_dataset("accuracy", data=np.array(accuracy_list2))
    grp.create_dataset("cossim", data=np.array(cossim_list2))
    grp.create_dataset("cossim_initB2", data=np.array(cossim_initB2_list))



## 読み込み

In [ ]:
# 読み込み
results = {}

file_name = f"result_{filename}.h5"
save_dir = f"./Dual_PC_result"
load_path = os.path.join(save_dir, file_name)

with h5py.File(save_path, "r") as f:
    energy_list1 = f["PC1"]["energy"][:]
    accuracy_list1    = f["PC1"]["accuracy"][:]
    cossim_list1 = f["PC1"]["cossim"][:]
    cossim_initB1_list = f["PC1"]["cossim_initB1"][:]

    energy_list2 = f["PC2"]["energy"][:]
    accuracy_list2 = f["PC2"]["accuracy"][:]
    cossim_list2 = f["PC2"]["cossim"][:]
    cossim_initB2_list = f["PC2"]["cossim_initB2"][:]


In [ ]:
# -----------------------------
# 複数トライアル平均と移動平均
# -----------------------------
def mean_std_across_trials(result_list):
    arr = np.array([np.array(r) for r in result_list])
    mean = np.mean(arr, axis=0)
    std  = np.std(arr, axis=0)
    return mean, std

def moving_average(x, w=10):
    return np.convolve(x, np.ones(w)/w, mode='valid')

# --- エネルギー ---
energy_mean_PC1, energy_std_PC1 = mean_std_across_trials(energy_list1)
energy_mean_PC2, energy_std_PC2 = mean_std_across_trials(energy_list2)

# --- 精度 ---
acc_mean_PC1, acc_std_PC1 = mean_std_across_trials(accuracy_list1)
acc_mean_PC2, acc_std_PC2 = mean_std_across_trials(accuracy_list2)

# --- コサイン類似度 ---
cos_PC1_mean, cos_PC1_std = mean_std_across_trials(cossim_list1)
cos_PC2_mean, cos_PC2_std = mean_std_across_trials(cossim_list2)
cosB1_mean,   cosB1_std = mean_std_across_trials(cossim_initB1_list)
cosB2_mean,   cosB2_std = mean_std_across_trials(cossim_initB2_list)

# 学習ステップ
steps_energy = np.arange(len(energy_mean_PC1)) * num_batch
steps_acc    = np.linspace(num_batch, num_step, len(acc_mean_PC1))
steps_cos    = steps_energy  # cos_simはstepごとに計算している前提

# -----------------------------
# エネルギー プロット
# -----------------------------
plt.figure()
plt.plot(steps_energy[len(steps_energy)-len(moving_average(energy_mean_PC1)):], moving_average(energy_mean_PC1), color='green', label='Energy PC1')
plt.plot(steps_energy[len(steps_energy)-len(moving_average(energy_mean_PC2)):], moving_average(energy_mean_PC2), color='black', label='Energy PC2')
plt.xlabel('Step')
plt.ylabel('Energy (moving average)')
plt.title('Energy over Learning')
plt.grid(True)
plt.legend()
plt.show()


# -----------------------------
# エネルギー プロット (log表示)
# -----------------------------
plt.figure()
plt.plot(steps_energy[len(steps_energy)-len(moving_average(energy_mean_PC1)):],
         moving_average(energy_mean_PC1), color='green', label='Energy PC1')
plt.plot(steps_energy[len(steps_energy)-len(moving_average(energy_mean_PC2)):],
         moving_average(energy_mean_PC2), color='black', label='Energy PC2')
plt.xlabel('Step')
plt.ylabel('Energy (moving average, log scale)')
plt.title('Energy over Learning')
plt.yscale('log')   # y軸をログスケールに変更
plt.grid(True, which="both")
plt.legend()
plt.show()


# -----------------------------
# 精度 プロット
# -----------------------------
plt.figure()
plt.plot(steps_acc, acc_mean_PC1*100, color='green', marker='o', label='Accuracy PC1')
plt.plot(steps_acc, acc_mean_PC2*100, color='black', marker='o', label='Accuracy PC2')
plt.xlabel('Step')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy over Learning')
plt.ylim(0, 100)  # 範囲固定
plt.grid(True)
plt.legend()
plt.show()

# -----------------------------
# FF-FB コサイン類似度 プロット
# -----------------------------
plt.figure()
plt.plot(steps_cos, cos_PC1_mean, color='green', label='Cosine PC1')
plt.plot(steps_cos, cos_PC2_mean, color='black', label='Cosine PC2')

plt.fill_between(steps_cos, cos_PC1_mean - cos_PC1_std, cos_PC1_mean + cos_PC1_std, alpha=0.2, color='gray')
plt.fill_between(steps_cos, cos_PC2_mean - cos_PC2_std, cos_PC2_mean + cos_PC2_std, alpha=0.2, color='gray')
plt.xlabel('Step')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity between FF and FB weights')
plt.ylim(-1, 1)  # 範囲固定
plt.grid(True)
plt.legend()
plt.show()

# -----------------------------
# inB-B コサイン類似度 プロット
# -----------------------------
plt.figure()
plt.plot(steps_cos, cosB1_mean, color='green', label='Cosine initB-B PC1')
plt.fill_between(steps_cos, cosB1_mean - cosB1_std, cosB1_mean + cosB1_std, alpha=0.2, color='green')
plt.plot(steps_cos, cosB2_mean, color='black', label='Cosine initB-B PC2')
plt.fill_between(steps_cos, cosB2_mean - cosB2_std, cosB2_mean + cosB2_std, alpha=0.2, color='gray')
plt.xlabel('Step')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity between initial and current B matrices')
plt.ylim(-1, 1)  # 範囲固定
plt.grid(True)
plt.legend()
plt.show()
